# EEG-to-3D Full Pipeline
**End-to-end training and inference on Colab Pro (A100/L4)**

This notebook:
1. Downloads data from Kaggle
2. Trains the EEG classifier (or loads existing checkpoint)
3. Trains the contrastive encoder with warm-start
4. Runs the full 7-stage EEG→3D pipeline
5. Computes evaluation metrics

## 0. Setup & Install

In [ ]:
# Mount Google Drive (optional — to persist outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the project repo from your fork
import os
REPO_DIR = '/content/EEG_to_3D'

if not os.path.exists(REPO_DIR):
    !git clone -b phase1-full-pipeline https://github.com/hamzaahmad-cp/EEG_to_3D.git {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')

os.chdir(REPO_DIR)
!pwd

In [ ]:
!pip install -q braindecode clip-by-openai diffusers transformers accelerate rembg open3d scikit-image pyyaml tqdm

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Download Data from Kaggle

In [ ]:
# Upload your kaggle.json
from google.colab import files
files.upload()  # Upload kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
DATA_DIR = f'{REPO_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{REPO_DIR}/checkpoints', exist_ok=True)

# Download EEG data
if not os.path.exists(f'{DATA_DIR}/eeg_55_95_std.pth'):
    !kaggle datasets download tariq9mehmood9/eeg-visual-classification-new -p /tmp/eeg_data
    !unzip -oq /tmp/eeg_data/eeg-visual-classification-new.zip -d {DATA_DIR}
    !rm -rf /tmp/eeg_data
    print('EEG data downloaded and extracted.')
else:
    print('EEG data already exists.')

# Download ImageNet-40 images
if not os.path.exists(f'{DATA_DIR}/imageNet_images'):
    !kaggle datasets download tariq9mehmood9/imagenet-40 -p /tmp/imagenet40
    !unzip -oq /tmp/imagenet40/imagenet-40.zip -d {DATA_DIR}
    !rm -rf /tmp/imagenet40
    print('ImageNet-40 images downloaded and extracted.')
else:
    print('ImageNet images already exist.')

# Download conformer baseline checkpoint
if not os.path.exists(f'{REPO_DIR}/checkpoints/eeg_classifier_best.pth'):
    !kaggle kernels output tariq9mehmood9/eeg-classification-baseline -p /tmp/conformer
    !cp /tmp/conformer/best_model.pth {REPO_DIR}/checkpoints/eeg_classifier_best.pth
    !rm -rf /tmp/conformer
    print('Classifier checkpoint downloaded.')
else:
    print('Classifier checkpoint already exists.')

In [ ]:
# Verify data
for f in ['eeg_55_95_std.pth', 'block_splits_by_image_all.pth', 'captions_with_bbox_data.pth', 'imagenet_class_labels.txt']:
    p = os.path.join(DATA_DIR, f)
    assert os.path.exists(p), f'Missing: {p}'
    print(f'  OK  {f} ({os.path.getsize(p)/1e6:.1f} MB)')

n_classes = len(os.listdir(os.path.join(DATA_DIR, 'imageNet_images')))
print(f'  OK  imageNet_images/ ({n_classes} classes)')
print(f'  OK  classifier checkpoint ({os.path.getsize(f"{REPO_DIR}/checkpoints/eeg_classifier_best.pth")/1e6:.1f} MB)')
print('\nAll data ready.')

## 2. Load Config & Data

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

with open('config/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Seed
seed = config['seed']
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

In [ ]:
from src.data.data_loader import CATVisDataLoader
from src.data.preprocessor import DataPreprocessor

data_loader = CATVisDataLoader(config)
df = data_loader.get_dataset_dataframe()
train_df, val_df, test_df = data_loader.get_train_val_test_splits(df)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'Classes: {len(data_loader.labels)}')
print(f'Unique images: {len(data_loader.image_to_path)}')

## 3. Stage 1: EEG Classifier — Test Existing Checkpoint

In [ ]:
from src.models.eeg_classifier import EEGClassifier

classifier = EEGClassifier(config).to(device)
classifier.model.load_state_dict(
    torch.load('checkpoints/eeg_classifier_best.pth', map_location=device))
classifier.eval()
print(f'Classifier loaded. Parameters: {sum(p.numel() for p in classifier.parameters()):,}')

In [ ]:
# Evaluate classifier on test set
preprocessor = DataPreprocessor(config)
_, _, test_cls_ds = preprocessor.create_classification_datasets(train_df, val_df, test_df)
test_cls_loader = torch.utils.data.DataLoader(test_cls_ds, batch_size=128, shuffle=False)

correct = {1: 0, 3: 0, 5: 0}
total = 0

classifier.eval()
with torch.no_grad():
    for eeg, label in tqdm(test_cls_loader, desc='Testing classifier'):
        eeg, label = eeg.to(device), label.to(device)
        outputs, _ = classifier(eeg)
        _, topk_preds = outputs.topk(5, dim=1)
        for k in [1, 3, 5]:
            correct[k] += (topk_preds[:, :k] == label.unsqueeze(1)).any(dim=1).sum().item()
        total += label.size(0)

print(f'\nClassifier Results ({total} test samples):')
for k in [1, 3, 5]:
    print(f'  Top-{k}: {100*correct[k]/total:.2f}%')

# CATVis baseline: Top-1=61.09%, Top-3=93.84%, Top-5=98.15%
print(f'\nCATVis baseline: Top-1=61.09%, Top-3=93.84%, Top-5=98.15%')

## 4. Stage 2: Train Contrastive Encoder

In [ ]:
import clip
from src.models.contrastive_encoder import ContrastiveEncoder, clip_style_contrastive_loss

TRAIN_CONTRASTIVE = True  # Set False to skip and load existing checkpoint

eeg_model = ContrastiveEncoder(config).to(device)

if TRAIN_CONTRASTIVE:
    # Warm-start from classifier backbone
    clf_state = torch.load('checkpoints/eeg_classifier_best.pth', map_location=device)
    model_state = eeg_model.model.state_dict()
    matched = 0
    for k, v in clf_state.items():
        if k in model_state and model_state[k].shape == v.shape:
            model_state[k] = v
            matched += 1
    eeg_model.model.load_state_dict(model_state)
    print(f'Warm-started: {matched}/{len(clf_state)} keys matched')

    # Frozen CLIP
    clip_model, _ = clip.load(config['contrastive_training']['clip_model'], device=device)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False

    # Data
    train_ct, val_ct, _ = preprocessor.create_contrastive_datasets(train_df, val_df, test_df)
    bs = config['contrastive_training']['batch_size']
    train_ct_loader = torch.utils.data.DataLoader(train_ct, batch_size=bs, shuffle=True)
    val_ct_loader = torch.utils.data.DataLoader(val_ct, batch_size=bs, shuffle=False)

    # Optimizer
    optimizer = torch.optim.Adam(eeg_model.parameters(), lr=config['contrastive_training']['learning_rate'])
    temperature = config['contrastive_training']['temperature']
    patience = config['contrastive_training']['patience']
    best_val_loss = float('inf')
    no_improve = 0

    print(f'\nTraining contrastive encoder for up to {config["contrastive_training"]["num_epochs"]} epochs...')
    print(f'Batch size: {bs}, Temperature: {temperature}, Patience: {patience}')

    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(config['contrastive_training']['num_epochs']):
        # Train
        eeg_model.train()
        train_losses = []
        for eeg_batch, text_batch in tqdm(train_ct_loader, desc=f'Epoch {epoch+1}', leave=False):
            eeg_batch = eeg_batch.to(device)
            with torch.no_grad():
                text_tokens = clip.tokenize(text_batch, truncate=True).to(device)
                text_embeds = clip_model.encode_text(text_tokens).float()
                text_embeds = F.normalize(text_embeds, dim=-1)
            eeg_embeds = eeg_model(eeg_batch)
            eeg_embeds = F.normalize(eeg_embeds, dim=-1)
            loss, _ = clip_style_contrastive_loss(eeg_embeds, text_embeds, temperature)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validate
        eeg_model.eval()
        val_losses = []
        with torch.no_grad():
            for eeg_batch, text_batch in val_ct_loader:
                eeg_batch = eeg_batch.to(device)
                text_tokens = clip.tokenize(text_batch, truncate=True).to(device)
                text_embeds = clip_model.encode_text(text_tokens).float()
                text_embeds = F.normalize(text_embeds, dim=-1)
                eeg_embeds = eeg_model(eeg_batch)
                eeg_embeds = F.normalize(eeg_embeds, dim=-1)
                loss, _ = clip_style_contrastive_loss(eeg_embeds, text_embeds, temperature)
                val_losses.append(loss.item())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        saved = ''
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            no_improve = 0
            torch.save(eeg_model.model.state_dict(), 'checkpoints/contrastive_model_best.pth')
            saved = ' [SAVED]'
        else:
            no_improve += 1

        print(f'Epoch {epoch+1:3d}  TrL={avg_train:.4f}  VaL={avg_val:.4f}  patience={no_improve}/{patience}{saved}')

        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}.')
            break

    # Reload best
    eeg_model.model.load_state_dict(
        torch.load('checkpoints/contrastive_model_best.pth', map_location=device))
    print(f'Best val loss: {best_val_loss:.4f}')

else:
    eeg_model.model.load_state_dict(
        torch.load('checkpoints/contrastive_model_best.pth', map_location=device))
    print('Loaded existing contrastive checkpoint.')

eeg_model.eval()
print('Contrastive encoder ready.')

In [ ]:
# Plot training curves
if TRAIN_CONTRASTIVE and history['train_loss']:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.plot(history['train_loss'], label='Train Loss')
    ax.plot(history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Contrastive Training')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Evaluate Retrieval (Recall@K)

In [ ]:
from src.pipeline.retrieval import TextRetrieval

retrieval = TextRetrieval(config, eeg_model, device)
retrieval.setup_retrieval_corpus(test_df)

# Build test EEG embeddings
from src.data.preprocessor import EEGTextDataset
test_ct_ds = EEGTextDataset(test_df)
test_ct_loader = torch.utils.data.DataLoader(test_ct_ds, batch_size=128, shuffle=False)

all_eeg_embeds = []
all_captions = []
eeg_model.eval()
with torch.no_grad():
    for eeg_batch, cap_batch in tqdm(test_ct_loader, desc='EEG embeddings'):
        eeg_batch = eeg_batch.to(device)
        emb = eeg_model(eeg_batch)
        emb = F.normalize(emb, dim=-1)
        all_eeg_embeds.append(emb.cpu())
        all_captions.extend(cap_batch)

all_eeg_embeds = torch.cat(all_eeg_embeds).to(device)
corpus_embeds = retrieval.unique_caption_embeds.to(device)

# Compute Recall@K
sim = all_eeg_embeds @ corpus_embeds.t()  # [N, M]
caption_to_idx = {cap: idx for idx, cap in enumerate(retrieval.unique_captions)}

N = len(all_captions)
hits = {1: 0, 5: 0, 10: 0}
for i in range(N):
    correct_idx = caption_to_idx[all_captions[i]]
    sorted_idx = torch.argsort(sim[i], descending=True)
    rank = (sorted_idx == correct_idx).nonzero(as_tuple=True)[0].item()
    for k in hits:
        if rank < k:
            hits[k] += 1

print(f'\nRetrieval Results ({N} test samples, {len(retrieval.unique_captions)} unique captions):')
for k in [1, 5, 10]:
    print(f'  Recall@{k}: {100*hits[k]/N:.2f}%')
print(f'\nCATVis baseline: R@1=5.36%, R@5=26.14%, R@10=45.27%')

## 6. Stages 3-5: Caption Retrieval + Prompt + Flux.1 Generation

In [ ]:
# Prepare test dataset for pipeline
test_dataset = preprocessor.create_pipeline_dataset(test_df)
labels_order = data_loader.raw_data['labels']

print(f'Test samples: {len(test_dataset)}')
print(f'Labels order: {labels_order[:5]}...')

In [ ]:
# Helpers
def is_caption_relevant(predicted_class, caption):
    class_words = predicted_class.lower().replace('-', ' ').replace('_', ' ').split()
    return any(len(w) > 3 and w in caption.lower() for w in class_words)

def create_enhanced_prompt(predicted_class, caption):
    use_caption = is_caption_relevant(predicted_class, caption)
    if use_caption:
        c = caption
        for src, dst in [('A group of','A single'),('group of','single'),
                         ('Several','One'),('several','one'),('Many','One'),('many','one'),
                         ('Two','One'),('two','one'),('Three','One'),('three','one'),
                         ('people',''),('persons',''),('men',''),('women','')]:
            c = c.replace(src, dst)
        prompt = (f'ONE single {predicted_class}, exactly one object, {c}, '
                  f'full body visible, complete anatomy, whole body in frame, '
                  f'solo subject only, no other objects, isolated on pure white background, '
                  f'centered, studio product photography, professional lighting, sharp focus, 8k')
    else:
        prompt = (f'ONE single {predicted_class}, exactly one object, '
                  f'full body visible, complete anatomy, whole body in frame, '
                  f'solo subject only, no other objects, isolated on pure white background, '
                  f'centered, studio product photography, professional lighting, sharp focus, 8k, photorealistic')
    return prompt, use_caption

In [ ]:
# Load Flux.1-schnell
from diffusers import FluxPipeline

flux_pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16)
flux_pipe.enable_model_cpu_offload()
print('Flux.1-schnell loaded.')

In [ ]:
import json, shutil, time
from pathlib import Path
from PIL import Image
from rembg import remove as rembg_remove

# Output dirs
out_root = Path('outputs/full_pipeline')
gen_dir = out_root / 'generated_images'
nobg_dir = out_root / 'generated_nobg'
gt_dir = out_root / 'ground_truth_images'
for d in [gen_dir, nobg_dir, gt_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Config
MAX_SAMPLES = 50  # Set to None for full test set
SEED = 42
FLUX_STEPS = 4

n_samples = len(test_dataset) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(test_dataset))
metadata = {}
meta_path = out_root / 'metadata.json'
if meta_path.exists():
    with open(meta_path) as f:
        metadata = json.load(f)

print(f'Generating for {n_samples} samples...')
t0 = time.time()

In [ ]:
for idx in tqdm(range(n_samples), desc='EEG -> Image'):
    key = f'sample_{idx:04d}'
    nobg_path = nobg_dir / f'{key}.png'
    
    # Skip if already done
    if nobg_path.exists() and key in metadata:
        continue

    sample = test_dataset[idx]
    eeg_data = sample['eeg'].unsqueeze(0).to(device)
    gt_idx = sample['label'].item() if torch.is_tensor(sample['label']) else sample['label']
    img_idx = sample['image'].item() if torch.is_tensor(sample['image']) else sample['image']

    # Stage 1-2: Classification
    with torch.no_grad():
        outputs, _ = classifier(eeg_data)
        pred_idx = outputs.argmax().item()

    predicted_class = config['class_prompts'][labels_order[pred_idx]]
    ground_truth_class = config['class_prompts'][labels_order[gt_idx]]

    # Stage 3: Caption retrieval
    retrieved = retrieval.retrieve_top_k_from_eeg(eeg_data.squeeze(0), k=1)
    caption = retrieved[0][0]

    # Stage 4: Prompt
    prompt, caption_used = create_enhanced_prompt(predicted_class, caption)

    # Stage 5: Flux.1 generation
    result = flux_pipe(
        prompt, guidance_scale=0.0, num_inference_steps=FLUX_STEPS,
        max_sequence_length=256,
        generator=torch.Generator('cpu').manual_seed(SEED + idx),
        height=1024, width=1024,
    )
    gen_image = result.images[0]
    gen_image.save(gen_dir / f'{key}.png')

    # Background removal
    image_no_bg = rembg_remove(gen_image)
    image_no_bg.save(nobg_path)

    # Save GT
    rel_path = data_loader.image_to_path.get(img_idx)
    if rel_path:
        src = Path(config['data']['root_dir']) / config['data']['imagenet_images'] / rel_path
        if src.exists():
            shutil.copy(src, gt_dir / f'{key}_gt.JPEG')

    metadata[key] = {
        'sample_idx': idx,
        'ground_truth_class': ground_truth_class,
        'predicted_class': predicted_class,
        'correct': predicted_class.lower() == ground_truth_class.lower(),
        'retrieved_caption': caption,
        'caption_used': caption_used,
    }

    # Save incrementally
    if (idx + 1) % 10 == 0:
        with open(meta_path, 'w') as f:
            json.dump(metadata, f, indent=2)

# Final save
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

elapsed = time.time() - t0
correct = sum(1 for v in metadata.values() if v.get('correct'))
print(f'\nDone in {elapsed/60:.1f} min')
print(f'Classification: {correct}/{len(metadata)} = {100*correct/len(metadata):.1f}%')
print(f'Generated images: {len(list(nobg_dir.glob("*.png")))}')

In [ ]:
# Show some results
import matplotlib.pyplot as plt

samples_to_show = list(metadata.items())[:8]
fig, axes = plt.subplots(2, 8, figsize=(24, 6))

for i, (key, meta) in enumerate(samples_to_show):
    if i >= 8: break
    # GT
    gt_path = gt_dir / f'{key}_gt.JPEG'
    if gt_path.exists():
        axes[0, i].imshow(Image.open(gt_path))
    axes[0, i].set_title(f'GT: {meta["ground_truth_class"]}', fontsize=8)
    axes[0, i].axis('off')
    
    # Generated
    gen_path = nobg_dir / f'{key}.png'
    if gen_path.exists():
        img = Image.open(gen_path).convert('RGB')
        axes[1, i].imshow(img)
    mark = 'Y' if meta['correct'] else 'X'
    axes[1, i].set_title(f'Pred: {meta["predicted_class"]} [{mark}]', fontsize=8)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Ground Truth', fontsize=10)
axes[1, 0].set_ylabel('Generated', fontsize=10)
plt.suptitle('EEG -> Image Generation Results', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Stage 6-7: Wonder3D + 3D Reconstruction

Wonder3D requires its own repo. Run this section only if Wonder3D is set up.

In [ ]:
# Clone Wonder3D (run once)
WONDER3D_DIR = Path('Wonder3D')
if not WONDER3D_DIR.exists():
    !git clone https://github.com/xxlong0/Wonder3D.git
    !cd Wonder3D && pip install -q -r requirements.txt
    print('Wonder3D cloned and installed.')
else:
    print('Wonder3D already exists.')

In [ ]:
import subprocess

RUN_WONDER3D = False  # Set True when Wonder3D is ready

if RUN_WONDER3D:
    wonder3d_input = WONDER3D_DIR / 'example_images'
    wonder3d_input.mkdir(parents=True, exist_ok=True)

    nobg_files = sorted(nobg_dir.glob('*.png'))
    print(f'Running Wonder3D on {len(nobg_files)} images...')

    for img_path in tqdm(nobg_files, desc='Wonder3D'):
        scene_name = img_path.stem
        scene_out = WONDER3D_DIR / 'outputs' / 'cropsize-192-cfg3.0' / scene_name
        if scene_out.exists():
            continue

        shutil.copy(img_path, wonder3d_input / img_path.name)
        try:
            subprocess.run([
                'python', 'test_mvdiffusion_seq.py',
                '--config', 'configs/mvdiffusion-joint-ortho-6views.yaml',
                f'validation_dataset.filepaths=[{img_path.name}]'
            ], cwd=str(WONDER3D_DIR), check=True, capture_output=True, timeout=300)
        except Exception as e:
            print(f'  Failed: {scene_name}: {e}')

    print('Wonder3D complete.')

In [ ]:
RUN_RECONSTRUCTION = False  # Set True after Wonder3D

if RUN_RECONSTRUCTION:
    from pixel2mesh_reconstruction import (
        load_views, space_carve, occupancy_to_mesh,
        assign_normals, assign_colors, make_o3d_mesh, save_mesh,
    )
    import open3d as o3d

    wonder3d_out = WONDER3D_DIR / 'outputs' / 'cropsize-192-cfg3.0'
    mesh_out_root = Path('outputs/meshes')

    scene_dirs = sorted([d for d in wonder3d_out.iterdir() if d.is_dir()])
    print(f'Reconstructing {len(scene_dirs)} meshes...')

    for scene_dir in tqdm(scene_dirs, desc='3D Reconstruction'):
        mesh_out = mesh_out_root / scene_dir.name
        if (mesh_out / 'mesh_carving.ply').exists():
            continue
        try:
            views = load_views(scene_dir)
            if not views:
                continue
            occ = space_carve(views, grid_res=128, orth_scale=1.05)
            if occ.sum() == 0:
                continue
            v, f = occupancy_to_mesh(occ, orth_scale=1.05, gaussian_sigma=1.0)
            nrm = assign_normals(v, views, orth_scale=1.05)
            clr = assign_colors(v, nrm, views, orth_scale=1.05)
            mesh = make_o3d_mesh(v, f, clr, nrm)
            mesh = mesh.filter_smooth_laplacian(number_of_iterations=3)
            mesh.compute_vertex_normals()
            save_mesh(mesh, mesh_out, 'mesh_carving')
        except Exception as e:
            print(f'  Failed: {scene_dir.name}: {e}')

    n_meshes = len(list(mesh_out_root.glob('*/mesh_carving.ply')))
    print(f'Reconstructed {n_meshes} meshes.')

## 8. Summary & Save to Drive

In [ ]:
# Summary
print('=' * 50)
print('  PIPELINE SUMMARY')
print('=' * 50)

if meta_path.exists():
    with open(meta_path) as f:
        metadata = json.load(f)
    correct = sum(1 for v in metadata.values() if v.get('correct'))
    total = len(metadata)
    print(f'  Samples processed: {total}')
    print(f'  Classification accuracy: {100*correct/total:.1f}%')
    print(f'  Generated images: {len(list(nobg_dir.glob("*.png")))}')

mesh_dir = Path('outputs/meshes')
if mesh_dir.exists():
    n_meshes = len(list(mesh_dir.glob('*/mesh_carving.ply')))
    print(f'  3D meshes: {n_meshes}')

print(f'\n  Outputs: {out_root}/')

In [ ]:
# Copy checkpoints and outputs to Google Drive for persistence
SAVE_TO_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/EEG_to_3D_outputs'

if SAVE_TO_DRIVE and os.path.exists('/content/drive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    
    # Save checkpoints
    os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
    for ckpt in Path('checkpoints').glob('*.pth'):
        shutil.copy(ckpt, f'{DRIVE_DIR}/checkpoints/{ckpt.name}')
    
    # Save metadata
    if meta_path.exists():
        shutil.copy(meta_path, f'{DRIVE_DIR}/metadata.json')
    
    print(f'Saved to {DRIVE_DIR}')
else:
    print('Google Drive not mounted or SAVE_TO_DRIVE=False')